# NB04 — Model Development

**Project:** The Access Gap: Using Machine Learning to Map Who Cannot Get Care in Canada and Why  
**Phase:** CRISP-DM Phase 5 — Modelling  
**Inputs:** `data/processed/X_train_smote_fe.npy`, `X_val_pp_fe.npy` (from NB03)  
**Outputs:** 5 fitted models, CV results, validation metrics, learning curves

---

## Objectives

Train and compare **5 classifiers** on the feature-engineered dataset from NB03:

| # | Model | Role | Class Imbalance Strategy |
|---|-------|------|-------------------------|
| 1 | **DummyClassifier** | Sanity baseline — must beat this | `strategy='stratified'` |
| 2 | **Logistic Regression** | Linear baseline — fast, interpretable | `class_weight='balanced'` |
| 3 | **Random Forest** | Ensemble of decision trees | `class_weight='balanced'` |
| 4 | **XGBoost** | Gradient boosting (fast, regularized) | `scale_pos_weight` |
| 5 | **LightGBM** | Leaf-wise gradient boosting (fastest) | `class_weight='balanced'` |

## Evaluation Strategy

- **Primary metric**: F1-score on the minority (at-risk) class  
- **Cross-validation**: 5-fold StratifiedKFold on `(X_train_pp_fe, y_train)` — *no SMOTE in CV folds* to prevent overoptimistic estimates  
- **Final fit**: On `(X_train_smote_fe, y_train_smote_fe)` — SMOTE-balanced training data  
- **Interim evaluation**: `X_val_pp_fe` only — **test set remains sealed until NB06**

In [ ]:
# ── Install / verify required packages ────────────────────────────────────────
import subprocess, sys
REQUIRED = {
    'pyarrow':         'pyarrow',
    'pandas':          'pandas',
    'numpy':           'numpy',
    'scikit-learn':    'sklearn',
    'imbalanced-learn':'imblearn',
    'xgboost':         'xgboost',
    'lightgbm':        'lightgbm',
    'matplotlib':      'matplotlib',
    'seaborn':         'seaborn',
    'joblib':          'joblib',
}
for pkg, imp in REQUIRED.items():
    try:
        __import__(imp)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'])
print('Packages OK')

---
## 0. Setup

In [ ]:
import sys, json, time, warnings
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    average_precision_score, precision_recall_curve,
    f1_score, precision_score, recall_score,
    confusion_matrix,
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
import config as cfg
from evaluation import (
    evaluate_model,
    find_optimal_threshold,
    plot_learning_curves,
    compare_models_table,
    plot_model_comparison,
)

plt.rcParams['figure.dpi'] = cfg.FIGURE_DPI
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

print(f'Project root : {PROJECT_ROOT}')
print(f'Random state : {cfg.RANDOM_STATE}')
print(f'CV folds     : {cfg.CV_FOLDS}')

---
## 1. Load Artifacts from NB03

> **Note:** Run NB03 completely before executing this notebook.  
> Required files: `X_train_smote_fe.npy`, `X_train_pp_fe.npy`, `X_val_pp_fe.npy`, `fe_config.json`.

In [ ]:
# ── Load preprocessed arrays ──────────────────────────────────────────────────
PROCESSED = cfg.DATA_PROCESSED

# ── Training: SMOTE version (used for final model fitting) ────────────────────
X_train_smote = np.load(PROCESSED / 'X_train_smote_fe.npy')
y_train_smote = np.load(PROCESSED / 'y_train_smote_fe.npy')

# ── Training: raw preprocessed (used for CV — no SMOTE leakage across folds) ──
X_train_pp    = np.load(PROCESSED / 'X_train_pp_fe.npy')

# ── Validation (interim evaluation — not for model selection, only monitoring) ─
X_val_pp      = np.load(PROCESSED / 'X_val_pp_fe.npy')

# ── TEST SET IS SEALED — do NOT load until NB06 ───────────────────────────────
# X_test_pp = np.load(PROCESSED / 'X_test_pp_fe.npy')   # <-- DO NOT UNCOMMENT

# ── Labels ────────────────────────────────────────────────────────────────────
y_train = pd.read_parquet(PROCESSED / 'y_train.parquet').squeeze().values
y_val   = pd.read_parquet(PROCESSED / 'y_val.parquet').squeeze().values

# ── Configs ───────────────────────────────────────────────────────────────────
with open(PROCESSED / 'fe_config.json') as f:
    fe_cfg = json.load(f)

with open(PROCESSED / 'feature_names_pp_fe.json') as f:
    feature_names = json.load(f)

print('DATA LOADED SUCCESSFULLY')
print(f'  X_train_smote : {X_train_smote.shape}  (SMOTE-balanced training)')
print(f'  X_train_pp    : {X_train_pp.shape}  (raw preprocessed training, for CV)')
print(f'  X_val_pp      : {X_val_pp.shape}  (validation)')
print(f'  y_train       : {y_train.shape}  (before SMOTE)')
print(f'  y_train_smote : {y_train_smote.shape}  (after SMOTE)')
print(f'  y_val         : {y_val.shape}')
print(f'  Feature names : {len(feature_names)}')

In [ ]:
# ── Class distribution summary ────────────────────────────────────────────────

splits_info = [
    ('Training (before SMOTE)', y_train),
    ('Training (after SMOTE)',  y_train_smote),
    ('Validation',              y_val),
]

print('CLASS DISTRIBUTION ACROSS SPLITS')
print(f'  {"Split":<28} {"Total":>7}  {"At-Risk":>8}  {"% At-Risk":>9}  {"Ratio":>7}')
print('  ' + '-' * 62)
for split_name, y in splits_info:
    total  = len(y)
    pos    = int(y.sum())
    neg    = total - pos
    pct    = pos / total * 100
    ratio  = neg / pos
    print(f'  {split_name:<28} {total:>7,}  {pos:>8,}  {pct:>9.2f}%  {ratio:>7.2f}:1')

print(f'\nBaseline F1 (if always predict "At Risk"): '
      f'{2 * y_val.mean() / (1 + y_val.mean()):.4f}')
print(f'Baseline F1 (stratified dummy, expected): ~{2 * y_val.mean()**2 / (2 * y_val.mean() - 1 + 1):.4f}')
print(f'\nTarget F1 >= 0.70 (minority class requirement)')

---
## 2. Model Definitions

### Design Choices

**Why these 5 models?**
- **DummyClassifier**: Establishes the floor — any real model must beat this. A model that doesn't beat a random baseline has learned nothing.
- **Logistic Regression**: Interpretable linear baseline. If LR performs well, the decision boundary is linear — simpler is better. Fast to train, no hyperparameter sensitivity.
- **Random Forest**: Robust ensemble baseline. Handles non-linearity, feature interactions, missing values. Serves as a strong mid-tier reference.
- **XGBoost**: State-of-the-art gradient boosting. Known to excel on tabular data competitions. Regularized, handles mixed feature types well.
- **LightGBM**: Leaf-wise boosting — faster training than XGBoost on large datasets, often competitive or better accuracy.

**Why `class_weight='balanced'` + SMOTE (not just one)?**  
SMOTE synthesizes new minority examples (structural resampling), while `class_weight='balanced'` adjusts the loss function. Combined, they provide two complementary signals to the model about minority class importance.

**Cross-validation note:**  
CV is run on `(X_train_pp, y_train)` — the non-SMOTE training data. Using `class_weight='balanced'` compensates. Running SMOTE *inside* each CV fold (via ImbPipeline) is the gold standard but adds complexity; the difference in estimates is minimal at this class ratio (~2.8:1).

In [ ]:
# ── Compute class weights for XGBoost (no class_weight param) ─────────────────
#
# scale_pos_weight = n_negative / n_positive in the TRAINING set (before SMOTE)
# This reweights the gradient for the positive class during boosting.

neg_count = int((y_train == 0).sum())
pos_count = int((y_train == 1).sum())
scale_pos_weight_train = neg_count / pos_count

# After SMOTE, the ratio changes — used only if fitting XGB on raw training
smote_neg = int((y_train_smote == 0).sum())
smote_pos = int((y_train_smote == 1).sum())
scale_pos_weight_smote = smote_neg / smote_pos

print(f'scale_pos_weight (before SMOTE, used in CV): {scale_pos_weight_train:.3f}')
print(f'scale_pos_weight (after  SMOTE, used in fit): {scale_pos_weight_smote:.3f}')

# ── Model zoo ────────────────────────────────────────────────────────────────
#
# NOTE: Two sets of models:
#   MODELS_CV   — used in cross-validation on raw (non-SMOTE) training data
#               → scale_pos_weight_train for XGBoost
#   MODELS_FIT  — used for final fitting on SMOTE training data
#               → scale_pos_weight_smote for XGBoost
#
# Hyperparameters here are sensible defaults, NOT tuned.
# NB05 (Optuna) tunes the top 3 models.

def make_models(xgb_scale_pos_weight):
    return OrderedDict([
        # ── 1. Dummy baseline ─────────────────────────────────────────────────
        ('Dummy (Stratified)', DummyClassifier(
            strategy='stratified',
            random_state=cfg.RANDOM_STATE,
        )),

        # ── 2. Logistic Regression ────────────────────────────────────────────
        # C=1.0 = moderate L2 regularization; lbfgs handles multiclass well
        # class_weight='balanced' scales loss by inverse class frequency
        ('Logistic Regression', LogisticRegression(
            C=1.0,
            max_iter=1000,
            class_weight='balanced',
            solver='lbfgs',
            random_state=cfg.RANDOM_STATE,
            n_jobs=cfg.N_JOBS,
        )),

        # ── 3. Random Forest ──────────────────────────────────────────────────
        # n_estimators=300 for stability; min_samples_leaf=10 for regularization
        # class_weight='balanced_subsample' = balanced per-bootstrap-sample
        ('Random Forest', RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=10,
            max_features='sqrt',
            class_weight='balanced_subsample',
            random_state=cfg.RANDOM_STATE,
            n_jobs=cfg.N_JOBS,
        )),

        # ── 4. XGBoost ────────────────────────────────────────────────────────
        # tree_method='hist': fast histogram-based algorithm
        # scale_pos_weight: compensates for class imbalance
        # subsample/colsample: regularization via stochastic subsampling
        ('XGBoost', XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=5,
            gamma=0.1,
            reg_alpha=0.1,
            reg_lambda=1.0,
            scale_pos_weight=xgb_scale_pos_weight,
            tree_method='hist',
            verbosity=0,
            random_state=cfg.RANDOM_STATE,
            n_jobs=cfg.N_JOBS,
        )),

        # ── 5. LightGBM ───────────────────────────────────────────────────────
        # num_leaves=63 (2^6-1) = expressive but not overfit on 40k samples
        # Leaf-wise growth: deeper, more accurate trees vs. XGBoost's level-wise
        ('LightGBM', LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            num_leaves=63,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_samples=20,
            reg_alpha=0.1,
            reg_lambda=1.0,
            class_weight='balanced',
            verbose=-1,
            random_state=cfg.RANDOM_STATE,
            n_jobs=cfg.N_JOBS,
        )),
    ])


# CV models: scale_pos_weight from raw y_train
MODELS_CV  = make_models(xgb_scale_pos_weight=scale_pos_weight_train)
# Fit models: scale_pos_weight from SMOTE y_train_smote
MODELS_FIT = make_models(xgb_scale_pos_weight=scale_pos_weight_smote)

print('Model zoo defined:')
for name in MODELS_FIT:
    print(f'  {name}')

---
## 3. Cross-Validation — Unbiased Model Comparison

**Why CV on non-SMOTE data?**  
SMOTE must never span CV fold boundaries. Applying SMOTE to the full training set before splitting leaks information from validation folds into training folds (synthetic points are interpolations of real points that may appear in the held-out fold). The correct approach is to apply SMOTE *inside* each fold (via `ImbPipeline`), but since the goal here is *model selection* rather than exact performance estimation, using `class_weight='balanced'` on the raw data gives a consistent, unbiased ranking signal.

**Metrics reported:** F1, PR-AUC, ROC-AUC, Recall, Precision

In [ ]:
# ── 5-Fold Stratified Cross-Validation ───────────────────────────────────────

cv_strategy = StratifiedKFold(
    n_splits=cfg.CV_FOLDS,
    shuffle=True,
    random_state=cfg.RANDOM_STATE,
)

# Multiple scoring metrics in one pass
CV_SCORING = {
    'f1':        'f1',
    'pr_auc':    'average_precision',
    'roc_auc':   'roc_auc',
    'recall':    'recall',
    'precision': 'precision',
}

cv_results = {}
print(f'Running {cfg.CV_FOLDS}-fold StratifiedKFold CV on X_train_pp (non-SMOTE)...')
print(f'  Training rows: {X_train_pp.shape[0]:,}   Features: {X_train_pp.shape[1]}')
print(f'  Minority class: {y_train.mean()*100:.2f}%')
print()

for name, model in MODELS_CV.items():
    t0 = time.time()
    # n_jobs=1: folds run sequentially; models use their own internal parallelism
    scores = cross_validate(
        model, X_train_pp, y_train,
        cv=cv_strategy,
        scoring=CV_SCORING,
        n_jobs=1,
        return_train_score=False,
        error_score=0.0,
    )
    elapsed = time.time() - t0

    cv_results[name] = {
        'f1_mean':         float(scores['test_f1'].mean()),
        'f1_std':          float(scores['test_f1'].std()),
        'pr_auc_mean':     float(scores['test_pr_auc'].mean()),
        'pr_auc_std':      float(scores['test_pr_auc'].std()),
        'roc_auc_mean':    float(scores['test_roc_auc'].mean()),
        'roc_auc_std':     float(scores['test_roc_auc'].std()),
        'recall_mean':     float(scores['test_recall'].mean()),
        'precision_mean':  float(scores['test_precision'].mean()),
        'time_s':          round(elapsed, 1),
        'f1_per_fold':     scores['test_f1'].tolist(),
    }

    print(f'  {name:<22} F1={scores["test_f1"].mean():.4f} ±{scores["test_f1"].std():.4f}'
          f'  PR-AUC={scores["test_pr_auc"].mean():.4f}'
          f'  [{elapsed:.1f}s]')

print('\nCV complete.')

In [ ]:
# ── CV Results Summary Table ──────────────────────────────────────────────────

cv_df = pd.DataFrame([
    {
        'Model':          name,
        'F1 (mean)':      res['f1_mean'],
        'F1 (std)':       res['f1_std'],
        'PR-AUC':         res['pr_auc_mean'],
        'ROC-AUC':        res['roc_auc_mean'],
        'Recall':         res['recall_mean'],
        'Precision':      res['precision_mean'],
        'Time (s)':       res['time_s'],
    }
    for name, res in cv_results.items()
]).sort_values('F1 (mean)', ascending=False).reset_index(drop=True)
cv_df.index = cv_df.index + 1

print('5-FOLD CROSS-VALIDATION RESULTS (training data, class_weight=balanced)')
print('=' * 90)
print(cv_df.to_string())
print('\nNote: All models must outperform DummyClassifier F1 — otherwise the model has learned nothing.')

# Identify top 3 for learning curves
# Exclude Dummy from ranking
top3_models_cv = (
    cv_df[cv_df['Model'] != 'Dummy (Stratified)']
    .head(3)['Model']
    .tolist()
)
print(f'\nTop 3 models for learning curves: {top3_models_cv}')

In [ ]:
# ── CV Results Visualization ──────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# ── 1. F1 bar chart with error bars ───────────────────────────────────────────
model_names_short = [
    n.replace('Dummy (Stratified)', 'Dummy').replace('Logistic Regression', 'Log. Reg.')
    .replace('Random Forest', 'Rand. Forest')
    for n in cv_df['Model']
]
colors_cv = ['#BDBDBD' if 'Dummy' in n else '#42A5F5' for n in cv_df['Model']]

bars = axes[0].bar(
    model_names_short, cv_df['F1 (mean)'],
    yerr=cv_df['F1 (std)'] * 1.96,  # 95% CI
    capsize=5, color=colors_cv, edgecolor='white',
    error_kw={'linewidth': 1.5, 'ecolor': 'dimgray'},
)
axes[0].axhline(0.70, color='#E53935', linestyle='--', linewidth=1.5,
                 label='Target F1 = 0.70')
for bar, val in zip(bars, cv_df['F1 (mean)']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')
axes[0].set_title('5-Fold CV F1-Score (At-Risk Class)\n(error bars = 95% CI)',
                   fontweight='bold')
axes[0].set_ylabel('F1-Score')
axes[0].set_ylim(0, 1.0)
axes[0].legend(fontsize=9)
axes[0].tick_params(axis='x', rotation=25)

# ── 2. PR-AUC vs ROC-AUC scatter ─────────────────────────────────────────────
for i, row in cv_df.iterrows():
    color = '#BDBDBD' if 'Dummy' in row['Model'] else '#1976D2'
    axes[1].scatter(row['ROC-AUC'], row['PR-AUC'], s=150, color=color, zorder=3)
    label_short = (row['Model'].replace('Dummy (Stratified)', 'Dummy')
                   .replace('Logistic Regression', 'LR')
                   .replace('Random Forest', 'RF'))
    axes[1].annotate(label_short,
                     (row['ROC-AUC'], row['PR-AUC']),
                     xytext=(5, 3), textcoords='offset points', fontsize=9)
axes[1].axhline(y_train.mean(), color='gray', linestyle=':', alpha=0.7,
                 label=f'PR-AUC baseline={y_train.mean():.3f}')
axes[1].set_xlabel('ROC-AUC')
axes[1].set_ylabel('PR-AUC')
axes[1].set_title('PR-AUC vs ROC-AUC\n(top-right = best)', fontweight='bold')
axes[1].legend(fontsize=9)

# ── 3. Per-fold F1 boxplot ────────────────────────────────────────────────────
fold_data    = [cv_results[n]['f1_per_fold'] for n in cv_df['Model']]
bp = axes[2].boxplot(
    fold_data,
    labels=model_names_short,
    patch_artist=True,
    medianprops=dict(color='black', linewidth=2),
)
for patch, color in zip(bp['boxes'], colors_cv):
    patch.set_facecolor(color)
axes[2].axhline(0.70, color='#E53935', linestyle='--', linewidth=1.5, label='Target = 0.70')
axes[2].set_title('F1 Distribution Across Folds\n(stability check)', fontweight='bold')
axes[2].set_ylabel('F1-Score per Fold')
axes[2].legend(fontsize=9)
axes[2].tick_params(axis='x', rotation=25)

plt.suptitle(f'{cfg.CV_FOLDS}-Fold Stratified CV — Model Comparison (Pre-Tuning Baselines)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(cfg.FIGURES_DIR / 'nb04_cv_comparison.png', bbox_inches='tight', dpi=cfg.FIGURE_DPI)
plt.show()

print('CV comparison saved.')

---
## 4. Fit Final Models on SMOTE Training Data

Each model is now fitted on the **full SMOTE-augmented training set** (`X_train_smote`, `y_train_smote`).  
This is the production-equivalent training — the model sees all available training examples.

After fitting, models are immediately serialised to `models/` so subsequent notebooks  
(NB05 tuning, NB06 final evaluation) can load them without re-training.

In [ ]:
# ── Fit all models on SMOTE training data ─────────────────────────────────────

fitted_models = {}
fit_times     = {}

print(f'Fitting 5 models on SMOTE training data...')
print(f'  X_train_smote: {X_train_smote.shape}   minority: {y_train_smote.mean()*100:.1f}%')
print()

for name, model in MODELS_FIT.items():
    t0 = time.time()
    model.fit(X_train_smote, y_train_smote)
    elapsed = time.time() - t0

    fitted_models[name] = model
    fit_times[name]     = round(elapsed, 1)

    # Quick sanity: check that model produces positive minority predictions
    y_prob_val   = model.predict_proba(X_val_pp)[:, 1]
    quick_f1     = f1_score(y_val, (y_prob_val >= 0.5).astype(int), pos_label=1, zero_division=0)
    quick_pr_auc = average_precision_score(y_val, y_prob_val)

    print(f'  {name:<22} fit={elapsed:5.1f}s   val_F1@0.5={quick_f1:.4f}   PR-AUC={quick_pr_auc:.4f}')

print('\nAll models fitted successfully.')

---
## 5. Validation Set Evaluation

Each model is evaluated on `X_val_pp` (the held-out validation set from NB02).  
This produces:
- Full evaluation dashboard (confusion matrix, ROC, PR, probability distribution, threshold scan)
- Saved figures for each model

> **The test set is not touched here.** Final performance numbers for the report  
> come from NB06, after threshold tuning is complete.

**Note on threshold:** Default threshold = 0.50 is used for the dashboard.  
Section 6 finds the optimal threshold per model by maximising F1 on the validation set.

In [ ]:
# ── Full evaluation dashboard for each model ──────────────────────────────────

val_results = {}  # Store results dicts for comparison

# File-safe name mapping for figure filenames
SAFE_NAMES = {
    'Dummy (Stratified)':  'dummy',
    'Logistic Regression': 'logistic_regression',
    'Random Forest':       'random_forest',
    'XGBoost':             'xgboost',
    'LightGBM':            'lightgbm',
}

for name, model in fitted_models.items():
    print(f'\n{"="*65}')
    print(f'  Evaluating: {name}')
    print(f'{"="*65}')

    safe  = SAFE_NAMES.get(name, name.lower().replace(' ', '_'))
    fpath = cfg.FIGURES_DIR / f'nb04_eval_{safe}.png'

    result = evaluate_model(
        model,
        X_val_pp, y_val,
        model_name=name,
        threshold=0.50,
        save_path=fpath,
    )
    val_results[name] = result

print('\nAll evaluation dashboards complete.')

In [ ]:
# ── Overlay ROC Curves ────────────────────────────────────────────────────────
#
# ROC-AUC measures ranking quality — how well the model separates classes
# regardless of threshold. A single overlaid plot makes ranking comparison easy.

fig, ax = plt.subplots(figsize=(9, 7))

OVERLAY_COLORS = {
    'Dummy (Stratified)':  '#BDBDBD',
    'Logistic Regression': '#FF9800',
    'Random Forest':       '#4CAF50',
    'XGBoost':             '#2196F3',
    'LightGBM':            '#9C27B0',
}

for name, model in fitted_models.items():
    y_prob  = model.predict_proba(X_val_pp)[:, 1]
    fpr, tpr, _ = roc_curve(y_val, y_prob)
    auc = roc_auc_score(y_val, y_prob)
    lw  = 1.5 if 'Dummy' in name else 2.5
    ax.plot(fpr, tpr, color=OVERLAY_COLORS[name],
            linewidth=lw, label=f'{name} (AUC={auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, linewidth=1, label='Random classifier')
ax.fill_between([0, 1], [0, 1], alpha=0.03, color='gray')
ax.set_xlabel('False Positive Rate (1 − Specificity)', fontsize=12)
ax.set_ylabel('True Positive Rate (Sensitivity / Recall)', fontsize=12)
ax.set_title('ROC Curves — All Models\n(Validation Set)',
             fontweight='bold', fontsize=13)
ax.legend(fontsize=10, loc='lower right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig(cfg.FIGURES_DIR / 'nb04_roc_overlay.png', bbox_inches='tight', dpi=cfg.FIGURE_DPI)
plt.show()

In [ ]:
# ── Overlay Precision-Recall Curves ──────────────────────────────────────────
#
# PR-AUC is more informative than ROC-AUC under class imbalance because it
# focuses entirely on the minority class — it does not factor in true negatives.
# Higher PR-AUC = better at finding at-risk individuals without too many false alarms.

fig, ax = plt.subplots(figsize=(9, 7))

for name, model in fitted_models.items():
    y_prob = model.predict_proba(X_val_pp)[:, 1]
    prec_arr, rec_arr, _ = precision_recall_curve(y_val, y_prob)
    ap = average_precision_score(y_val, y_prob)
    lw = 1.5 if 'Dummy' in name else 2.5
    ax.plot(rec_arr, prec_arr, color=OVERLAY_COLORS[name],
            linewidth=lw, label=f'{name} (AP={ap:.3f})')

# Baseline: a model with no skill predicts minority class rate everywhere
baseline_pr = y_val.mean()
ax.axhline(baseline_pr, color='gray', linestyle='--', alpha=0.7,
            label=f'No-skill baseline ({baseline_pr:.3f})')
ax.set_xlabel('Recall (Sensitivity — fraction of at-risk caught)', fontsize=12)
ax.set_ylabel('Precision (fraction of predicted at-risk that are truly at-risk)', fontsize=12)
ax.set_title('Precision-Recall Curves — All Models\n(Validation Set)',
             fontweight='bold', fontsize=13)
ax.legend(fontsize=10, loc='upper right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
plt.tight_layout()
plt.savefig(cfg.FIGURES_DIR / 'nb04_pr_overlay.png', bbox_inches='tight', dpi=cfg.FIGURE_DPI)
plt.show()

---
## 6. Optimal Threshold Analysis

The default 0.50 threshold assumes equal misclassification costs, which is rarely  
optimal for imbalanced classification. A lower threshold (e.g., 0.35) increases  
**recall** (catching more at-risk individuals) at the cost of **precision** (more false alarms).

**For a public health screening tool**, recall is typically prioritised over precision:  
missing an at-risk individual (false negative) is more costly than over-referring (false positive).

Here we find the F1-maximising threshold. NB05 (tuning) can explore recall-prioritised thresholds.

In [ ]:
# ── Find optimal threshold for each model ─────────────────────────────────────

print('THRESHOLD OPTIMIZATION (F1-maximising on validation set)')
print('=' * 75)
print(f'  {"Model":<22} {"Default F1":>10} {"Optimal Thr":>12} {"Optimal F1":>11} {"Recall":>8} {"Precision":>10}')
print('  ' + '-' * 73)

threshold_results = {}

for name, model in fitted_models.items():
    opt_thr, opt_f1 = find_optimal_threshold(model, X_val_pp, y_val, metric='f1')

    y_prob     = model.predict_proba(X_val_pp)[:, 1]
    y_pred_def = (y_prob >= 0.50).astype(int)
    y_pred_opt = (y_prob >= opt_thr).astype(int)

    f1_default = f1_score(y_val, y_pred_def, pos_label=1, zero_division=0)
    rec_opt    = recall_score(y_val, y_pred_opt, pos_label=1, zero_division=0)
    prec_opt   = precision_score(y_val, y_pred_opt, pos_label=1, zero_division=0)

    threshold_results[name] = {
        'default_f1':    round(f1_default, 4),
        'optimal_thr':   round(opt_thr, 3),
        'optimal_f1':    round(opt_f1, 4),
        'recall_at_opt': round(rec_opt, 4),
        'prec_at_opt':   round(prec_opt, 4),
    }

    print(f'  {name:<22} {f1_default:>10.4f} {opt_thr:>12.3f} {opt_f1:>11.4f} {rec_opt:>8.4f} {prec_opt:>10.4f}')

# Update val_results with optimal threshold info
for name in val_results:
    val_results[name].update({
        'optimal_threshold': threshold_results[name]['optimal_thr'],
        'optimal_f1':        threshold_results[name]['optimal_f1'],
    })

In [ ]:
# ── Threshold scan visualization for top models ───────────────────────────────

# Show threshold vs F1 curves for non-dummy models
plot_models_thr = [n for n in fitted_models if 'Dummy' not in n]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes_flat = axes.flatten()
thresholds_scan = np.arange(0.10, 0.91, 0.01)

for ax, name in zip(axes_flat, plot_models_thr):
    model     = fitted_models[name]
    y_prob    = model.predict_proba(X_val_pp)[:, 1]

    f1s  = [f1_score(y_val, (y_prob >= t).astype(int), pos_label=1, zero_division=0)
            for t in thresholds_scan]
    recs = [recall_score(y_val,    (y_prob >= t).astype(int), pos_label=1, zero_division=0)
            for t in thresholds_scan]
    pres = [precision_score(y_val, (y_prob >= t).astype(int), pos_label=1, zero_division=0)
            for t in thresholds_scan]

    opt_thr = threshold_results[name]['optimal_thr']
    opt_f1  = threshold_results[name]['optimal_f1']

    ax.plot(thresholds_scan, f1s,  color='#7B1FA2', linewidth=2.0, label='F1')
    ax.plot(thresholds_scan, recs, color='#E53935', linewidth=1.5, linestyle='--', label='Recall')
    ax.plot(thresholds_scan, pres, color='#1976D2', linewidth=1.5, linestyle='--', label='Precision')
    ax.axvline(opt_thr, color='black', linestyle=':', linewidth=1.5,
               label=f'Optimal thr={opt_thr:.2f} (F1={opt_f1:.3f})')
    ax.axvline(0.50, color='gray', linestyle=':', linewidth=1.0, alpha=0.7, label='Default 0.50')
    ax.axhline(0.70, color='#F57C00', linestyle=':', linewidth=1.0, alpha=0.7, label='Target F1=0.70')
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Decision Threshold')
    ax.set_ylabel('Metric Score')
    ax.legend(fontsize=8)
    ax.set_xlim([0.10, 0.90])
    ax.set_ylim([0, 1.05])

plt.suptitle('Threshold vs F1 / Recall / Precision — Top Models (Validation Set)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(cfg.FIGURES_DIR / 'nb04_threshold_analysis.png', bbox_inches='tight', dpi=cfg.FIGURE_DPI)
plt.show()

print('Threshold analysis saved.')

---
## 7. Learning Curves — Top 3 Models

Learning curves answer: *"Would adding more training data help?"*

| Pattern | Diagnosis | Fix |
|---------|-----------|-----|
| Large train-val **gap** (>0.10) | High variance / overfitting | More data, regularize, reduce complexity |
| Both curves **low and flat** | High bias / underfitting | More complex model, add features |
| Curves **converging, val still rising** | More data would help | Collect more data |
| Curves **converging, val plateau** | Good fit, data-saturated | Move to tuning |

These are run on **raw training data** (`X_train_pp, y_train`) to correctly diagnose  
learning behaviour. Learning curves use internal 5-fold CV (from `evaluation.py`).

In [ ]:
# ── Learning curves for top 3 CV models ──────────────────────────────────────
#
# Uses evaluation.py → plot_learning_curves()
# Input: raw (non-SMOTE) training data + labels, with class_weight='balanced'
# Purpose: diagnose overfitting/underfitting BEFORE hyperparameter tuning (NB05)

lc_results = {}

# Build fresh model instances for LC (same config as MODELS_CV, which uses class_weight)
# CV models are already used — rebuild equivalent from MODELS_CV
for name in top3_models_cv:
    print(f'\nLearning curves: {name}...')
    t0 = time.time()

    model_lc = MODELS_CV[name]  # Uses class_weight='balanced' config
    safe     = SAFE_NAMES.get(name, name.lower().replace(' ', '_'))

    lc_info = plot_learning_curves(
        model_lc,
        X_train_pp, y_train,
        model_name=name,
        cv=cfg.CV_FOLDS,
        scoring='f1',
        save_path=cfg.FIGURES_DIR / f'nb04_lc_{safe}.png',
    )
    elapsed = time.time() - t0

    lc_results[name] = lc_info
    print(f'  Train score: {lc_info["train_mean_final"]:.4f}')
    print(f'  Val score:   {lc_info["val_mean_final"]:.4f}')
    print(f'  Gap:         {lc_info["gap"]:.4f}  [{elapsed:.1f}s]')

print('\nLearning curves complete.')

In [ ]:
# ── Learning Curve Interpretation Summary ─────────────────────────────────────

print('LEARNING CURVE DIAGNOSTICS')
print('=' * 65)
print(f'  {"Model":<22} {"Train F1":>10} {"Val F1":>9} {"Gap":>7} {"Diagnosis":>12}')
print('  ' + '-' * 63)

for name in top3_models_cv:
    lc = lc_results[name]
    train_f = lc['train_mean_final']
    val_f   = lc['val_mean_final']
    gap     = lc['gap']

    if gap > 0.10:
        diagnosis = 'Overfit'
    elif val_f < 0.60:
        diagnosis = 'Underfit'
    else:
        diagnosis = 'Good fit'

    print(f'  {name:<22} {train_f:>10.4f} {val_f:>9.4f} {gap:>7.4f} {diagnosis:>12}')

print()
print('Interpretation:')
print('  - Overfitting models will benefit most from regularisation tuning in NB05')
print('  - Good-fit models with val F1 < 0.70 need hyperparameter tuning to close the gap')
print('  - Converging curves suggest more SMOTE samples or features would help less than tuning')

---
## 8. Model Comparison Summary

Consolidating CV results and validation evaluation into a single ranked comparison table.  
This table directly informs which models to prioritise for hyperparameter tuning in NB05.

**Primary criterion for NB05 selection:** Validation PR-AUC (most imbalance-robust metric)

In [ ]:
# ── Comprehensive comparison table ───────────────────────────────────────────

print('FULL MODEL COMPARISON — CV + Validation Results')
print('=' * 100)

# Build combined comparison dataframe
comparison_rows = []
for name in MODELS_FIT:
    cv  = cv_results[name]
    val = val_results[name]
    thr = threshold_results[name]

    comparison_rows.append({
        'Model':             name,
        # CV metrics
        'CV F1':             round(cv['f1_mean'], 4),
        'CV F1 ±std':        round(cv['f1_std'], 4),
        'CV PR-AUC':         round(cv['pr_auc_mean'], 4),
        'CV ROC-AUC':        round(cv['roc_auc_mean'], 4),
        # Validation metrics (threshold=0.50)
        'Val F1 @0.50':      val['f1'],
        'Val PR-AUC':        val['pr_auc'],
        'Val ROC-AUC':       val['roc_auc'],
        'Val Recall @0.50':  val['recall'],
        # Optimal threshold
        'Opt. Threshold':    thr['optimal_thr'],
        'Opt. F1':           thr['optimal_f1'],
        'Recall @ Opt.Thr':  thr['recall_at_opt'],
        # Fit time
        'Fit Time (s)':      fit_times[name],
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df = comparison_df.sort_values('Val PR-AUC', ascending=False).reset_index(drop=True)
comparison_df.index = comparison_df.index + 1

# Display transposed for readability
print(comparison_df.T.to_string())

# Identify best model by Val PR-AUC (excluding Dummy)
best_row = comparison_df[comparison_df['Model'] != 'Dummy (Stratified)'].iloc[0]
print(f'\nBEST MODEL by Val PR-AUC: {best_row["Model"]}')
print(f'  Val PR-AUC:   {best_row["Val PR-AUC"]:.4f}')
print(f'  Val F1 @opt:  {best_row["Opt. F1"]:.4f}')
print(f'  Opt threshold:{best_row["Opt. Threshold"]:.3f}')

In [ ]:
# ── Visual comparison: Validation metrics ─────────────────────────────────────

# Use evaluation.py plot_model_comparison with the val_results dicts
# Ensure all required keys present
plot_results_list = list(val_results.values())
plot_model_comparison(
    plot_results_list,
    save_path=cfg.FIGURES_DIR / 'nb04_model_comparison.png',
)

print('Model comparison chart saved.')

In [ ]:
# ── CV vs Validation F1 comparison (consistency check) ───────────────────────
#
# Large gap between CV F1 and validation F1 would indicate overfitting to the
# training distribution or leakage in the CV fold setup.

fig, ax = plt.subplots(figsize=(11, 6))

model_names_plot = list(cv_results.keys())
x = np.arange(len(model_names_plot))
width = 0.30

cv_f1s  = [cv_results[n]['f1_mean']   for n in model_names_plot]
val_f1s = [val_results[n]['f1']       for n in model_names_plot]
opt_f1s = [threshold_results[n]['optimal_f1'] for n in model_names_plot]

bars1 = ax.bar(x - width,   cv_f1s,  width, label='5-Fold CV F1 (class_weight=balanced)',
               color='#42A5F5', edgecolor='white', alpha=0.9)
bars2 = ax.bar(x,            val_f1s, width, label='Validation F1 (threshold=0.50)',
               color='#FFA726', edgecolor='white', alpha=0.9)
bars3 = ax.bar(x + width,   opt_f1s, width, label='Validation F1 (optimal threshold)',
               color='#66BB6A', edgecolor='white', alpha=0.9)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        if h > 0.02:
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.004,
                    f'{h:.3f}', ha='center', fontsize=7.5, fontweight='bold')

ax.axhline(0.70, color='#E53935', linestyle='--', linewidth=1.5, label='Target F1 = 0.70')
short_names = [n.replace('Dummy (Stratified)', 'Dummy').replace('Logistic Regression', 'LR')
               .replace('Random Forest', 'RF') for n in model_names_plot]
ax.set_xticks(x)
ax.set_xticklabels(short_names, fontsize=11)
ax.set_ylabel('F1-Score (At-Risk Class)', fontsize=12)
ax.set_title('CV F1 vs Validation F1 vs Optimal-Threshold F1\n'
             '(Gap between CV and Validation indicates generalisation behaviour)',
             fontweight='bold', fontsize=12)
ax.legend(fontsize=9)
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig(cfg.FIGURES_DIR / 'nb04_cv_vs_val_f1.png', bbox_inches='tight', dpi=cfg.FIGURE_DPI)
plt.show()

print('\nConsistency check (CV F1 vs Val F1 gap):')
for name in model_names_plot:
    gap = cv_results[name]['f1_mean'] - val_results[name]['f1']
    flag = '  [!] Large gap — possible overfit' if abs(gap) > 0.08 else ''
    print(f'  {name:<22}: CV={cv_results[name]["f1_mean"]:.4f}, Val={val_results[name]["f1"]:.4f}, gap={gap:+.4f}{flag}')

---
## 9. Save Models & Results

In [ ]:
# ── Save all fitted models ────────────────────────────────────────────────────

for name, model in fitted_models.items():
    safe      = SAFE_NAMES.get(name, name.lower().replace(' ', '_'))
    out_path  = cfg.MODELS_DIR / f'{safe}.joblib'
    joblib.dump(model, out_path)
    print(f'  Saved: {out_path.name}')

print('All models saved.')

In [ ]:
# ── Save results to JSON & CSV for downstream notebooks ───────────────────────

# CV results JSON
cv_results_serialisable = {
    name: {k: v for k, v in res.items() if k != 'raw'}
    for name, res in cv_results.items()
}
with open(PROCESSED / 'cv_results_nb04.json', 'w') as f:
    json.dump(cv_results_serialisable, f, indent=2)

# Validation results JSON
with open(PROCESSED / 'val_results_nb04.json', 'w') as f:
    json.dump(val_results, f, indent=2)

# Threshold results JSON
with open(PROCESSED / 'threshold_results_nb04.json', 'w') as f:
    json.dump(threshold_results, f, indent=2)

# Combined comparison CSV
comparison_df.to_csv(PROCESSED / 'model_comparison_nb04.csv', index=True)

# NB05 selection: top 3 non-dummy models by Val PR-AUC
top3_for_tuning = (
    comparison_df[comparison_df['Model'] != 'Dummy (Stratified)']
    .head(3)['Model']
    .tolist()
)
nb04_manifest = {
    'top3_for_tuning':  top3_for_tuning,
    'best_model':       top3_for_tuning[0],
    'model_safe_names': SAFE_NAMES,
    'n_features':       int(X_train_pp.shape[1]),
    'n_train_smote':    int(X_train_smote.shape[0]),
    'n_val':            int(X_val_pp.shape[0]),
    'minority_pct_val': float(y_val.mean() * 100),
}
with open(PROCESSED / 'nb04_manifest.json', 'w') as f:
    json.dump(nb04_manifest, f, indent=2)

print('Results saved:')
for fname in ['cv_results_nb04.json', 'val_results_nb04.json',
              'threshold_results_nb04.json', 'model_comparison_nb04.csv',
              'nb04_manifest.json']:
    size = (PROCESSED / fname).stat().st_size / 1024
    print(f'  {fname:<42}: {size:.1f} KB')

print(f'\nTop 3 models selected for NB05 tuning: {top3_for_tuning}')

In [ ]:
# ── Final artifact inventory ──────────────────────────────────────────────────

print('NB04 ARTIFACT INVENTORY')
print('=' * 65)

print('\nModels saved (models/):')
for name in fitted_models:
    safe = SAFE_NAMES.get(name, name.lower().replace(' ', '_'))
    p    = cfg.MODELS_DIR / f'{safe}.joblib'
    if p.exists():
        print(f'  {p.name:<35}: {p.stat().st_size/1024:.1f} KB')

print('\nResults saved (data/processed/):')
for fname in ['cv_results_nb04.json', 'val_results_nb04.json',
              'threshold_results_nb04.json', 'model_comparison_nb04.csv',
              'nb04_manifest.json']:
    p = PROCESSED / fname
    if p.exists():
        print(f'  {fname:<42}: {p.stat().st_size/1024:.1f} KB')

print('\nFigures saved (reports/figures/):')
for f in sorted(cfg.FIGURES_DIR.glob('nb04_*.png')):
    print(f'  {f.name}')

---
## 10. NB04 Summary

### Results

| Notebook | Model Selection | Evaluation |
|----------|----------------|------------|
| NB04 (this) | 5-fold CV on training set | Validation set only |
| NB05 | Optuna tuning on top 3 | Cross-validated |
| NB06 | Final tuned models | **Test set (first use)** |

### Modelling Decisions

| Decision | Choice | Rationale |
|----------|--------|----------|
| CV data | Raw (non-SMOTE) training | Avoids SMOTE leakage across folds |
| Class imbalance in CV | `class_weight='balanced'` | Compensates for ~2.8:1 ratio without SMOTE |
| Final training | SMOTE-augmented data | Provides resampled minority examples |
| Primary metric | F1 (minority class) | Not misled by class imbalance |
| Threshold | F1-maximising scan | Default 0.50 is suboptimal for imbalanced data |

### Anti-Leakage Checklist

- [x] No test set loaded or evaluated in this notebook
- [x] CV run on raw training data only (no SMOTE leakage)
- [x] Models fitted on SMOTE training data (already constructed in NB03)
- [x] Optimal thresholds found using validation set only
- [x] All models saved before NB06 test evaluation

### Next Steps

- **NB05**: Hyperparameter tuning with Optuna — top 3 models selected above  
  Focus: tune learning rate, depth, regularization, threshold  
  Target: push F1 ≥ 0.70 on validation, generalise to test set
- **NB06**: Final evaluation — first and only use of `X_test_pp_fe.npy`  
  SHAP explainability + policy recommendations